# Dual-Stage LSTM-Autoencoder Feature Selection Comparison

Memory-efficient CSE-CICIDS2018 experiment using four feature-selection methods, 80/20 benign-per-day temporal split, non-overlapping windows, and a residual dual-stage LSTM autoencoder.

In [14]:
import gc
import glob
import json
import os
import pickle
import shutil
import tempfile
import time
import warnings

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.feature_selection import RFE, mutual_info_classif
from sklearn.metrics import (
    accuracy_score,
    auc,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler
from mrmr import mrmr_classif

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def log_step(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


Using device: cpu


In [15]:
# --- Configuration ---
DATASET_DIR = "/home/dhani/AnomalyIDS/dataset"
ARTIFACTS_DIR = "/home/dhani/AnomalyIDS/dual-stage-ae/artifacts"
TEMP_ROOT = "/tmp/dual_stage_ae"

TIME_STEPS = 10
WINDOW_STRIDE = TIME_STEPS  # non-overlapping windows
TRAIN_BENIGN_RATIO = 0.8
HIDDEN_DIM = 64
LEARNING_RATE = 0.001
DROPOUT = 0.2
BATCH_SIZE = 64
EPOCHS_STAGE1 = 10
EPOCHS_STAGE2 = 10
K_FEATURES = 10
CSV_CHUNK_SIZE = 50000
PARQUET_BATCH_SIZE = 32768
FS_SAMPLE_PER_CLASS = 20000
RFE_SAMPLE_PER_CLASS = 5000
RF_ESTIMATORS = 25
RFE_RF_ESTIMATORS = 10
RFE_STEP = 5
THRESHOLD_PERCENTILE = 95
CORRELATION_THRESHOLD = 0.9


In [16]:
def safe_name(name):
    return name.lower().replace(" ", "_").replace("-", "_")


def optimize_df(df):
    numeric_cols = [col for col in df.columns if col != "Label"]
    df[numeric_cols] = df[numeric_cols].astype("float32")
    return df


def get_shared_feature_columns(dataset_dir):
    log_step(f"Scanning shared feature columns: {dataset_dir}")
    csv_files = sorted(glob.glob(os.path.join(dataset_dir, "*.csv")))
    if not csv_files:
        raise ValueError("No CSV files found in dataset directory.")
    reference_columns = pd.read_csv(csv_files[0], nrows=0).columns.tolist()
    shared_columns = set(reference_columns)
    for file in csv_files[1:]:
        shared_columns &= set(pd.read_csv(file, nrows=0).columns.tolist())
    feature_columns = [
        col for col in reference_columns
        if col in shared_columns and col not in {"Label", "Timestamp"}
    ]
    if not feature_columns:
        raise ValueError("No shared feature columns were found.")
    log_step(f"Shared feature columns ready: {len(feature_columns)}")
    return feature_columns


def clean_chunk(df, feature_columns):
    cleaned = df[feature_columns + ["Label"]].copy()
    cleaned[feature_columns] = cleaned[feature_columns].apply(pd.to_numeric, errors="coerce")
    cleaned.replace([np.inf, -np.inf], np.nan, inplace=True)
    valid_mask = cleaned[feature_columns].notna().all(axis=1) & cleaned["Label"].notna()
    cleaned = cleaned.loc[valid_mask].copy()
    if cleaned.empty:
        return cleaned
    return optimize_df(cleaned)


def count_valid_benign_rows(file_path, feature_columns, chunk_size=CSV_CHUNK_SIZE):
    log_step(f"Counting valid benign rows: {os.path.basename(file_path)}")
    total = 0
    usecols = feature_columns + ["Label"]
    for chunk in pd.read_csv(file_path, usecols=usecols, chunksize=chunk_size, low_memory=False):
        chunk = clean_chunk(chunk, feature_columns)
        total += int((chunk["Label"] == "Benign").sum())
    log_step(f"Valid benign rows in {os.path.basename(file_path)}: {total}")
    return total


def update_sample_buffer(buffer_df, chunk_df, max_rows, rng):
    if chunk_df.empty:
        return buffer_df
    combined = chunk_df.copy() if buffer_df is None or buffer_df.empty else pd.concat([buffer_df, chunk_df], ignore_index=True)
    if len(combined) <= max_rows:
        return combined.reset_index(drop=True)
    sampled_idx = rng.choice(len(combined), size=max_rows, replace=False)
    return combined.iloc[sampled_idx].reset_index(drop=True)


In [17]:
def get_representative_sample(dataset_dir, sample_size_per_class=FS_SAMPLE_PER_CLASS, feature_columns=None):
    log_step("Sampling bounded data for feature selection")
    benign_sample = None
    attack_sample = None
    rng = np.random.default_rng(SEED)
    csv_files = sorted(glob.glob(os.path.join(dataset_dir, "*.csv")))
    if feature_columns is None:
        feature_columns = get_shared_feature_columns(dataset_dir)
    usecols = feature_columns + ["Label"]

    for file in csv_files:
        log_step(f"Sampling file: {os.path.basename(file)}")
        for chunk in pd.read_csv(file, usecols=usecols, chunksize=CSV_CHUNK_SIZE, low_memory=False):
            chunk = clean_chunk(chunk, feature_columns)
            if chunk.empty:
                continue
            benign_chunk = chunk[chunk["Label"] == "Benign"]
            attack_chunk = chunk[chunk["Label"] != "Benign"]
            benign_sample = update_sample_buffer(benign_sample, benign_chunk, sample_size_per_class, rng)
            attack_sample = update_sample_buffer(attack_sample, attack_chunk, sample_size_per_class, rng)
            del chunk, benign_chunk, attack_chunk
            gc.collect()

    sampled_frames = []
    if benign_sample is not None and not benign_sample.empty:
        sampled_frames.append(benign_sample)
    if attack_sample is not None and not attack_sample.empty:
        sampled_frames.append(attack_sample)
    if not sampled_frames:
        raise ValueError("No valid rows found for feature selection sampling.")

    combined = pd.concat(sampled_frames, ignore_index=True)
    X = combined[feature_columns].astype("float32")
    y = (combined["Label"] != "Benign").astype("int8").values
    log_step(f"Feature selection sample shape: {X.shape}")
    log_step(f"Shared feature columns: {len(feature_columns)}")
    if X.isna().any().any():
        raise ValueError("NaNs remain in feature selection sample.")
    return X, y


def get_balanced_subset(X, y, max_per_class):
    rng = np.random.default_rng(SEED)
    idx_parts = []
    for cls in [0, 1]:
        cls_idx = np.flatnonzero(y == cls)
        take = min(len(cls_idx), max_per_class)
        if take == 0:
            continue
        if len(cls_idx) > take:
            cls_idx = rng.choice(cls_idx, size=take, replace=False)
        idx_parts.append(cls_idx)
    if not idx_parts:
        raise ValueError("Balanced subset could not be created.")
    idx = np.concatenate(idx_parts)
    rng.shuffle(idx)
    return X.iloc[idx].reset_index(drop=True), y[idx]


def compute_mi_scores(X, y):
    return pd.Series(mutual_info_classif(X, y, random_state=SEED), index=X.columns, dtype="float64")


def prune_correlated_features(X, y, threshold=CORRELATION_THRESHOLD):
    log_step(f"Pruning correlated features at threshold={threshold}")
    mi_scores = compute_mi_scores(X, y)
    variances = X.var()
    corr_matrix = X.corr().abs()
    columns = X.columns.tolist()
    dropped = set()
    dropped_details = []

    for i, left_col in enumerate(columns):
        if left_col in dropped:
            continue
        for j in range(i + 1, len(columns)):
            right_col = columns[j]
            if right_col in dropped:
                continue
            corr_value = corr_matrix.loc[left_col, right_col]
            if pd.isna(corr_value) or corr_value <= threshold:
                continue
            left_mi, right_mi = float(mi_scores[left_col]), float(mi_scores[right_col])
            left_var, right_var = float(variances[left_col]), float(variances[right_col])
            if left_mi < right_mi:
                drop_col, keep_col = left_col, right_col
            elif right_mi < left_mi:
                drop_col, keep_col = right_col, left_col
            elif left_var < right_var:
                drop_col, keep_col = left_col, right_col
            elif right_var < left_var:
                drop_col, keep_col = right_col, left_col
            else:
                drop_col, keep_col = right_col, left_col
            dropped.add(drop_col)
            dropped_details.append({
                "kept_feature": keep_col,
                "dropped_feature": drop_col,
                "correlation": float(corr_value),
                "kept_mi": float(mi_scores[keep_col]),
                "dropped_mi": float(mi_scores[drop_col]),
                "kept_variance": float(variances[keep_col]),
                "dropped_variance": float(variances[drop_col]),
            })
            if drop_col == left_col:
                break

    pruned_columns = [col for col in columns if col not in dropped]
    log_step(f"Correlation pruning dropped {len(dropped_details)} features; {len(pruned_columns)} remain")
    return pruned_columns, dropped_details, mi_scores


In [18]:
def select_features_mi(X, y, k):
    log_step("Selecting features: Mutual Information")
    k = min(k, X.shape[1])
    mi_scores = compute_mi_scores(X, y)
    selected = X.columns[np.argsort(mi_scores.values)[-k:]].tolist()
    log_step(f"MI selected: {selected}")
    return selected


def select_features_rf(X, y, k):
    log_step("Selecting features: RF Importance")
    from sklearn.ensemble import RandomForestClassifier
    k = min(k, X.shape[1])
    rf = RandomForestClassifier(
        n_estimators=RF_ESTIMATORS,
        random_state=SEED,
        n_jobs=1,
        max_depth=12,
        min_samples_leaf=5,
    )
    rf.fit(X, y)
    selected = X.columns[np.argsort(rf.feature_importances_)[-k:]].tolist()
    log_step(f"RF selected: {selected}")
    return selected


def select_features_rfe(X, y, k):
    log_step("Selecting features: RFE")
    from sklearn.ensemble import RandomForestClassifier
    k = min(k, X.shape[1])
    X_small, y_small = get_balanced_subset(X, y, RFE_SAMPLE_PER_CLASS)
    log_step(f"RFE subset shape: {X_small.shape}")
    rf = RandomForestClassifier(
        n_estimators=RFE_RF_ESTIMATORS,
        random_state=SEED,
        n_jobs=1,
        max_depth=10,
        min_samples_leaf=10,
    )
    rfe = RFE(estimator=rf, n_features_to_select=k, step=RFE_STEP)
    rfe.fit(X_small, y_small)
    selected = X_small.columns[rfe.support_].tolist()
    log_step(f"RFE selected: {selected}")
    return selected


def mrmr_relevance_mi(X, y):
    return compute_mi_scores(X, np.asarray(y))


def select_features_mrmr(X, y, k):
    log_step("Selecting features: mRMR")
    k = min(k, X.shape[1])
    selected = mrmr_classif(
        X=X,
        y=pd.Series(y, name="Label"),
        K=k,
        relevance=mrmr_relevance_mi,
        redundancy="c",
        n_jobs=1,
        show_progress=False,
    )
    selected = list(selected)
    log_step(f"mRMR selected: {selected}")
    return selected


methods = {
    "Mutual Information": select_features_mi,
    "RF Importance": select_features_rf,
    "RFE": select_features_rfe,
    "mRMR": select_features_mrmr,
}


In [19]:
def write_parquet_chunk(writer, df, path):
    if df.empty:
        return writer
    table = pa.Table.from_pandas(df, preserve_index=False)
    if writer is None:
        writer = pq.ParquetWriter(path, table.schema, compression="snappy")
    writer.write_table(table)
    return writer


def preprocess_and_save_parquet(dataset_dir, selected_features, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    csv_files = sorted(glob.glob(os.path.join(dataset_dir, "*.csv")))
    parquet_files = {"train": [], "test": []}
    split_audit = []
    cols_to_keep = selected_features + ["Label"]

    log_step(f"Preprocessing 80/20 benign split to {output_dir}")
    for file in csv_files:
        fname = os.path.basename(file)
        log_step(f"Preprocessing file: {fname}")
        train_path = os.path.join(output_dir, f"train_{fname}.parquet")
        test_path = os.path.join(output_dir, f"test_{fname}.parquet")
        total_benign = count_valid_benign_rows(file, selected_features)
        train_cap = int(total_benign * TRAIN_BENIGN_RATIO)
        benign_seen = 0
        train_benign = 0
        test_benign = 0
        attack_test = 0
        train_writer = None
        test_writer = None

        for chunk in pd.read_csv(file, usecols=cols_to_keep, chunksize=CSV_CHUNK_SIZE, low_memory=False):
            chunk = clean_chunk(chunk, selected_features)
            if chunk.empty:
                continue
            chunk = chunk[cols_to_keep]
            benign = chunk[chunk["Label"] == "Benign"]
            attack = chunk[chunk["Label"] != "Benign"]

            if not benign.empty:
                can_take = max(0, train_cap - benign_seen)
                take_train = min(len(benign), can_take)
                if take_train > 0:
                    train_writer = write_parquet_chunk(train_writer, benign.iloc[:take_train], train_path)
                test_benign_part = benign.iloc[take_train:]
                benign_seen += len(benign)
                train_benign += take_train
                test_benign += len(test_benign_part)
            else:
                test_benign_part = benign

            if not attack.empty:
                attack_test += len(attack)

            test_part = pd.concat([test_benign_part, attack], axis=0).sort_index()
            if not test_part.empty:
                test_writer = write_parquet_chunk(test_writer, test_part, test_path)

            del chunk, benign, attack, test_benign_part, test_part
            gc.collect()

        if train_writer is not None:
            train_writer.close()
        else:
            pd.DataFrame(columns=cols_to_keep).to_parquet(train_path, index=False)
        if test_writer is not None:
            test_writer.close()
        else:
            pd.DataFrame(columns=cols_to_keep).to_parquet(test_path, index=False)

        parquet_files["train"].append(train_path)
        parquet_files["test"].append(test_path)
        split_audit.append({
            "file": fname,
            "total_valid_benign": int(total_benign),
            "train_benign": int(train_benign),
            "test_benign": int(test_benign),
            "attack_test": int(attack_test),
            "train_benign_ratio": float(train_benign / total_benign) if total_benign else 0.0,
        })
        log_step(f"Done {fname}: train_benign={train_benign}, test_benign={test_benign}, attack_test={attack_test}")
        gc.collect()

    return parquet_files, pd.DataFrame(split_audit)


def fit_streaming_scaler(file_list, selected_features):
    log_step("Fitting streaming StandardScaler")
    scaler = StandardScaler()
    seen_batch = False
    for file in file_list:
        log_step(f"Scaler file: {os.path.basename(file)}")
        parquet_file = pq.ParquetFile(file)
        for batch in parquet_file.iter_batches(batch_size=PARQUET_BATCH_SIZE, columns=selected_features):
            df = batch.to_pandas()
            if df.empty:
                continue
            scaler.partial_fit(df[selected_features].to_numpy(dtype=np.float32, copy=True))
            seen_batch = True
            del df
    if not seen_batch:
        raise ValueError("No train data available to fit StandardScaler.")
    log_step("Scaler ready")
    return scaler


def count_window_sequences(n_rows, time_steps=TIME_STEPS, stride=WINDOW_STRIDE):
    if n_rows < time_steps:
        return 0
    return 1 + (n_rows - time_steps) // stride


def count_sequences_from_parquet(file_list, time_steps=TIME_STEPS, stride=WINDOW_STRIDE):
    total = 0
    for file in file_list:
        total += count_window_sequences(pq.ParquetFile(file).metadata.num_rows, time_steps, stride)
    return total


In [20]:
class StreamingLSTMDataset(torch.utils.data.IterableDataset):
    def __init__(self, file_list, selected_features, time_steps, stride=1, scaler=None, is_test=False):
        super().__init__()
        self.file_list = file_list
        self.selected_features = selected_features
        self.time_steps = time_steps
        self.stride = stride
        self.scaler = scaler
        self.is_test = is_test

    def __iter__(self):
        for file in self.file_list:
            parquet_file = pq.ParquetFile(file)
            buffer = None
            labels_buffer = None

            for batch in parquet_file.iter_batches(batch_size=PARQUET_BATCH_SIZE, columns=self.selected_features + ["Label"]):
                df = batch.to_pandas()
                if df.empty:
                    continue
                data_vals = df[self.selected_features].to_numpy(dtype=np.float32, copy=True)
                if self.scaler is not None:
                    data_vals = self.scaler.transform(data_vals).astype(np.float32)
                labels = (df["Label"] != "Benign").astype(np.int8).values if self.is_test else None

                if buffer is not None and len(buffer) > 0:
                    data_vals = np.concatenate([buffer, data_vals], axis=0)
                    if self.is_test:
                        labels = np.concatenate([labels_buffer, labels])

                limit = len(data_vals) - self.time_steps + 1
                if limit > 0:
                    for i in range(0, limit, self.stride):
                        seq = data_vals[i : i + self.time_steps]
                        if self.is_test:
                            label = 1 if labels[i : i + self.time_steps].sum() > 0 else 0
                            yield torch.from_numpy(seq), torch.tensor(label, dtype=torch.long)
                        else:
                            yield torch.from_numpy(seq), torch.from_numpy(seq)
                    next_start = ((limit - 1) // self.stride + 1) * self.stride
                else:
                    next_start = 0

                if next_start < len(data_vals):
                    buffer = data_vals[next_start:].copy()
                    labels_buffer = labels[next_start:].copy() if self.is_test else None
                else:
                    buffer = None
                    labels_buffer = None

                del df, data_vals, labels

            del buffer, labels_buffer
            gc.collect()


In [21]:
class LSTMAutoencoder(nn.Module):
    def __init__(self, num_features, time_steps, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.time_steps = time_steps
        self.encoder_lstm = nn.LSTM(num_features, hidden_dim, batch_first=True)
        self.encoder_dropout = nn.Dropout(dropout)
        self.decoder_lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.decoder_dropout = nn.Dropout(dropout)
        self.output_layer = nn.Linear(hidden_dim, num_features)

    def forward(self, x):
        _, (hn, _) = self.encoder_lstm(x)
        latent = self.encoder_dropout(hn[-1])
        repeated = latent.unsqueeze(1).repeat(1, self.time_steps, 1)
        decoded, _ = self.decoder_lstm(repeated)
        decoded = self.decoder_dropout(decoded)
        return self.output_layer(decoded)


class DualStageAutoencoder(nn.Module):
    def __init__(self, num_features, time_steps, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.stage1 = LSTMAutoencoder(num_features, time_steps, hidden_dim, dropout)
        self.stage2 = LSTMAutoencoder(num_features, time_steps, hidden_dim, dropout)

    def forward(self, x):
        recon1 = self.stage1(x)
        residual = torch.abs(x - recon1)
        recon_residual = self.stage2(residual)
        return recon1, recon_residual


def dual_stage_score(model, x):
    recon1, recon_residual = model(x)
    residual = torch.abs(x - recon1)
    stage1_error = residual.mean(dim=(1, 2))
    stage2_error = torch.abs(residual - recon_residual).mean(dim=(1, 2))
    return stage1_error + stage2_error


In [22]:
def train_stage1(model, parquet_files, selected_features, scaler):
    log_step("Training Stage 1 autoencoder")
    criterion = nn.L1Loss()
    optimizer = torch.optim.Adam(model.stage1.parameters(), lr=LEARNING_RATE)
    for epoch in range(1, EPOCHS_STAGE1 + 1):
        model.stage1.train()
        epoch_loss = 0.0
        total = 0
        dataset = StreamingLSTMDataset(parquet_files["train"], selected_features, TIME_STEPS, WINDOW_STRIDE, scaler=scaler)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE)
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            recon = model.stage1(bx)
            loss = criterion(recon, by)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * bx.size(0)
            total += bx.size(0)
        if total == 0:
            raise ValueError("No train sequences generated for stage 1.")
        log_step(f"Stage 1 epoch {epoch}/{EPOCHS_STAGE1} | loss={epoch_loss / total:.6f}")


def train_stage2(model, parquet_files, selected_features, scaler):
    log_step("Training Stage 2 residual autoencoder")
    for param in model.stage1.parameters():
        param.requires_grad = False
    model.stage1.eval()
    criterion = nn.L1Loss()
    optimizer = torch.optim.Adam(model.stage2.parameters(), lr=LEARNING_RATE)
    for epoch in range(1, EPOCHS_STAGE2 + 1):
        model.stage2.train()
        epoch_loss = 0.0
        total = 0
        dataset = StreamingLSTMDataset(parquet_files["train"], selected_features, TIME_STEPS, WINDOW_STRIDE, scaler=scaler)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE)
        for bx, _ in loader:
            bx = bx.to(device)
            with torch.no_grad():
                residual = torch.abs(bx - model.stage1(bx))
            optimizer.zero_grad()
            recon_residual = model.stage2(residual)
            loss = criterion(recon_residual, residual)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * bx.size(0)
            total += bx.size(0)
        if total == 0:
            raise ValueError("No train sequences generated for stage 2.")
        log_step(f"Stage 2 epoch {epoch}/{EPOCHS_STAGE2} | loss={epoch_loss / total:.6f}")


def write_scores(model, file_list, selected_features, scaler, errors_path, labels_path=None):
    log_step(f"Scoring {'test' if labels_path is not None else 'train'} sequences")
    is_test = labels_path is not None
    total_sequences = count_sequences_from_parquet(file_list, TIME_STEPS, WINDOW_STRIDE)
    if total_sequences == 0:
        raise ValueError("No sequences generated for scoring.")
    log_step(f"Total sequences to score: {total_sequences}")
    errors = np.memmap(errors_path, dtype="float32", mode="w+", shape=(total_sequences,))
    labels = np.memmap(labels_path, dtype="int8", mode="w+", shape=(total_sequences,)) if is_test else None
    idx = 0
    model.eval()
    with torch.no_grad():
        dataset = StreamingLSTMDataset(file_list, selected_features, TIME_STEPS, WINDOW_STRIDE, scaler=scaler, is_test=is_test)
        loader = DataLoader(dataset, batch_size=BATCH_SIZE)
        for bx, by in loader:
            bx = bx.to(device)
            score = dual_stage_score(model, bx).cpu().numpy()
            end = idx + len(score)
            errors[idx:end] = score
            if is_test:
                labels[idx:end] = by.numpy()
            idx = end
    errors.flush()
    log_step("Scoring done")
    if is_test:
        labels.flush()
        return errors, labels
    return errors


def safe_roc_auc(y_true, scores):
    return float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) > 1 else float("nan")


def safe_pr_auc(y_true, scores):
    if len(np.unique(y_true)) < 2:
        return float("nan")
    precision, recall, _ = precision_recall_curve(y_true, scores)
    return float(auc(recall, precision))


In [23]:
def train_and_evaluate(method_name, selected_features, dataset_dir, pruned_feature_columns=None, dropped_correlated_features=None):
    print()
    log_step(f">>> Starting dual-stage pipeline for: {method_name}")
    pruned_feature_columns = pruned_feature_columns or selected_features
    dropped_correlated_features = dropped_correlated_features or []
    run_name = safe_name(method_name)
    os.makedirs(TEMP_ROOT, exist_ok=True)
    output_dir = tempfile.mkdtemp(prefix=f"{run_name}_", dir=TEMP_ROOT)
    artifact_dir = os.path.join(ARTIFACTS_DIR, run_name)
    os.makedirs(artifact_dir, exist_ok=True)

    parquet_files, split_audit_df = preprocess_and_save_parquet(dataset_dir, selected_features, output_dir)
    log_step("Split audit")
    print(split_audit_df.to_string(index=False))
    scaler = fit_streaming_scaler(parquet_files["train"], selected_features)

    log_step("Building dual-stage model")
    model = DualStageAutoencoder(len(selected_features), TIME_STEPS, HIDDEN_DIM, DROPOUT).to(device)
    train_stage1(model, parquet_files, selected_features, scaler)
    train_stage2(model, parquet_files, selected_features, scaler)

    train_errors_path = os.path.join(output_dir, f"train_errors_{run_name}.bin")
    train_errors = write_scores(model, parquet_files["train"], selected_features, scaler, train_errors_path)
    threshold = float(np.percentile(train_errors, THRESHOLD_PERCENTILE))
    log_step(f"Train anomaly threshold (P{THRESHOLD_PERCENTILE}): {threshold:.6f}")

    test_errors_path = os.path.join(output_dir, f"test_errors_{run_name}.bin")
    test_labels_path = os.path.join(output_dir, f"test_labels_{run_name}.bin")
    test_errors, test_labels = write_scores(model, parquet_files["test"], selected_features, scaler, test_errors_path, test_labels_path)

    y_true = np.asarray(test_labels)
    scores = np.asarray(test_errors)
    y_pred = (scores > threshold).astype(np.int8)
    log_step("Computing metrics")
    metrics = {
        "Selected Features": ", ".join(selected_features),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "ROC-AUC": safe_roc_auc(y_true, scores),
        "PR-AUC": safe_pr_auc(y_true, scores),
    }

    model_path = os.path.join(artifact_dir, "model.pt")
    scaler_path = os.path.join(artifact_dir, "scaler.pkl")
    metadata_path = os.path.join(artifact_dir, "metadata.json")
    log_step("Saving artifacts")
    torch.save({
        "model_state_dict": model.state_dict(),
        "selected_features": selected_features,
        "threshold": threshold,
        "time_steps": TIME_STEPS,
        "window_stride": WINDOW_STRIDE,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,
    }, model_path)
    with open(scaler_path, "wb") as f:
        pickle.dump(scaler, f)
    with open(metadata_path, "w") as f:
        json.dump({
            "method_name": method_name,
            "selected_features": selected_features,
            "pruned_feature_columns": pruned_feature_columns,
            "dropped_correlated_features": dropped_correlated_features,
            "train_benign_ratio": TRAIN_BENIGN_RATIO,
            "window_stride": WINDOW_STRIDE,
            "threshold": threshold,
            "threshold_percentile": THRESHOLD_PERCENTILE,
            "time_steps": TIME_STEPS,
            "hidden_dim": HIDDEN_DIM,
            "dropout": DROPOUT,
            "batch_size": BATCH_SIZE,
            "epochs_stage1": EPOCHS_STAGE1,
            "epochs_stage2": EPOCHS_STAGE2,
            "k_features": K_FEATURES,
            "split_audit": split_audit_df.to_dict(orient="records"),
            "metrics": metrics,
        }, f, indent=2)
    log_step(f"Saved model to {model_path}")
    log_step(f"Saved scaler to {scaler_path}")

    del train_errors, test_errors, test_labels, y_true, scores, y_pred, model
    gc.collect()
    shutil.rmtree(output_dir, ignore_errors=True)
    log_step(f"Finished method: {method_name}")
    return metrics


In [24]:
# --- Self-checks ---
def run_self_checks():
    with tempfile.TemporaryDirectory() as tmpdir:
        data = pd.DataFrame({
            "f1": np.arange(13, dtype=np.float32),
            "f2": np.arange(13, dtype=np.float32) * 2,
            "Label": ["Benign"] * 10 + ["Attack"] * 3,
        })
        data.to_csv(os.path.join(tmpdir, "toy.csv"), index=False)
        outdir = os.path.join(tmpdir, "parquet")
        parquet_files, audit = preprocess_and_save_parquet(tmpdir, ["f1", "f2"], outdir)
        row = audit.iloc[0]
        assert row["train_benign"] == 8
        assert row["test_benign"] == 2
        assert row["attack_test"] == 3
        train_ds = StreamingLSTMDataset(parquet_files["train"], ["f1", "f2"], time_steps=4, stride=4)
        test_ds = StreamingLSTMDataset(parquet_files["test"], ["f1", "f2"], time_steps=4, stride=4, is_test=True)
        assert sum(1 for _ in train_ds) == 2
        test_items = list(test_ds)
        assert len(test_items) == 1
        assert int(test_items[0][1]) == 1

    assert count_window_sequences(12, time_steps=10, stride=10) == 1
    assert count_window_sequences(20, time_steps=10, stride=10) == 2
    dummy_model = DualStageAutoencoder(K_FEATURES, TIME_STEPS, HIDDEN_DIM, DROPOUT)
    dummy_x = torch.randn(2, TIME_STEPS, K_FEATURES)
    recon1, recon2 = dummy_model(dummy_x)
    assert recon1.shape == dummy_x.shape
    assert recon2.shape == dummy_x.shape
    log_step("Self-checks passed")


run_self_checks()


[23:49:03] Preprocessing 80/20 benign split to /tmp/tmpb0a2h_ra/parquet
[23:49:03] Preprocessing file: toy.csv
[23:49:03] Counting valid benign rows: toy.csv
[23:49:03] Valid benign rows in toy.csv: 10
[23:49:03] Done toy.csv: train_benign=8, test_benign=2, attack_test=3
[23:49:05] Self-checks passed


In [25]:
# --- Execution Loop ---
log_step("Starting full dual-stage comparison")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
shared_feature_columns = get_shared_feature_columns(DATASET_DIR)
log_step(f"Using {len(shared_feature_columns)} shared feature columns across all files")

X_fs, y_fs = get_representative_sample(
    DATASET_DIR,
    sample_size_per_class=FS_SAMPLE_PER_CLASS,
    feature_columns=shared_feature_columns,
)
pruned_feature_columns, dropped_correlated_features, pruning_mi_scores = prune_correlated_features(
    X_fs,
    y_fs,
    threshold=CORRELATION_THRESHOLD,
)
X_fs_pruned = X_fs[pruned_feature_columns].copy()
final_results = {}

for name, func in methods.items():
    selected_features = func(X_fs_pruned, y_fs, K_FEATURES)
    log_step(f"{name} selected features: {selected_features}")
    final_results[name] = train_and_evaluate(
        name,
        selected_features,
        DATASET_DIR,
        pruned_feature_columns=pruned_feature_columns,
        dropped_correlated_features=dropped_correlated_features,
    )
    gc.collect()

results_df = pd.DataFrame(final_results).T
print()
log_step("=== FINAL DUAL-STAGE COMPARISON ===")
print(results_df)


[23:49:05] Starting full dual-stage comparison
[23:49:05] Scanning shared feature columns: /home/dhani/AnomalyIDS/dataset
[23:49:05] Shared feature columns ready: 78
[23:49:05] Using 78 shared feature columns across all files
[23:49:05] Sampling bounded data for feature selection
[23:49:05] Sampling file: Friday-02-03-2018_TrafficForML_CICFlowMeter.csv
[23:49:33] Sampling file: Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
[23:49:59] Sampling file: Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
[23:50:21] Sampling file: Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
[23:53:15] Sampling file: Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
[23:53:27] Sampling file: Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
[23:53:47] Sampling file: Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
[23:54:06] Sampling file: Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
[23:54:24] Sampling file: Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
[23:54:42] Sampling file: Wednesday-28-